In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
# import pprint

# A set of fixed input
a_value = 1.0
n_psa = 40000
seed = 2025

sample_size_now = 397
sample_size_new = 616

wtp = 150000

max_cycle = 200
# Define the cycle range
cycle_range = np.arange(0, max_cycle + 1)  # 0 to 200 inclusive

def get_discount_factor(cycle_range, dr=0.03, cycles_per_year=16):
    """
    Calculates the discount factor for each cycle based on the discount rate.

    Args:
        cycle_range (array-like): The range of cycles (e.g., np.arange(0, 201)).
        dr (float): The yearly discount rate (default is 0.03, or 3%).
        cycles_per_year (int): The number of cycles in a year (default is 16).

    Returns:
        np.ndarray: An array of discount factors corresponding to the cycle range.
    """
    # Convert yearly discount rate to per-cycle discount rate
    discount_rate_cycle = (1 + dr) ** (1 / cycles_per_year) - 1

    # Calculate the discount factor for each cycle
    discount_factor = 1 / (1 + discount_rate_cycle) ** cycle_range

    return discount_factor

discount_factor = get_discount_factor(cycle_range)

# Hazard ratio for Combo versus Chemo
hr_params = {
    "pfs": {"hr": 0.49, "low": 0.38, "high": 0.63},
    "os": {"hr": 0.60, "low": 0.45, "high": 0.79}
}

# Define the utility dictionary
utility = {
    "stable": 0.75,  # Utility for stable state
    "prog": 0.59     # Utility for progression state
}

# Define the cost dictionary
cost = {
    # Monitoring costs
    "monitoring_stable": 464.85 * 3,
    "monitoring_prog": 1075.49 * 3,

    # Parameters for drug costs
    "surface": 1.86,
    "admin_first": 158.7,
    "admin_sub": 33.6,
    "terminal": 16441.83,
    "price_premetrxed": 7.49,
    "price_cisplatin": 0.18,
    "price_carboplatin": 0.05,
    "dose_premetrxed": 500,
    "dose_cisplatin": 75,
    "dose_carboplatin": 550,
    "dose_drug": 200,

    # Calculated costs
    "intro_cost": (
        1.86 * 500 * 7.49 +
        0.277 * 1.86 * 75 * 0.18 +
        (1 - 0.277) * 550 * 0.05
    ),
    "maintain_cost_chemo": 1.86 * 500 * 7.49
}

In [ ]:
value_based_price_loc = "/content/drive/MyDrive/Colab Notebooks/01_Research/01_Local_Confirmatory_RCT/03_output/02_value_price"
value_based_price = pd.read_csv(f"{value_based_price_loc}/value_price.csv").iloc[0,0]

point_price = value_based_price * (1-0.1)

surv_params_path = "/content/drive/MyDrive/Colab Notebooks/01_Research/01_Local_Confirmatory_RCT/03_output/01_surv_params"
surv_params = pd.read_csv(f"{surv_params_path}/surv_params.csv")
case_params = {
    row["case_name"]: {
        "intercept": row["intercept"],
        "log_scale": row["log_scale"]
    }
    for _, row in surv_params.iterrows()
}

In [ ]:
def hr_log_transform(hr: float,
                     hr_lower: float,
                     hr_upper: float) -> dict:
    """
    Compute the mean and standard deviation of the log-transformed hazard ratio (HR).

    Parameters:
    hr (float): Hazard ratio estimate.
    ci_lower (float): Lower bound of the confidence interval.
    ci_upper (float): Upper bound of the confidence interval.

    Returns:
    tuple: (mean of log(HR), standard deviation of log(HR))
    """
    log_hr = np.log(hr)
    log_ci_lower = np.log(hr_lower)
    log_ci_upper = np.log(hr_upper)

    # Approximate standard deviation using CI width
    sigma_log_hr_old = (log_ci_upper - log_ci_lower) / (2 * norm.ppf(1 - 0.05 / 2))
    # sigma_log_hr_updated = np.sqrt(sigma_log_hr_old**2 / a_value)

    # return {"mu_log_hr": log_hr,
    #         "sigma_log_hr": sigma_log_hr_old,
    #         "sigma_log_hr_updated": sigma_log_hr_updated}

    return {"mu_log_hr": log_hr,
            "sigma_log_hr": sigma_log_hr_old}

In [ ]:
def get_hr_vec(
    hr_params: dict = hr_params,
    n_psa: int = n_psa,
    seed_for_prior: int = seed
) -> dict:
    """
    Generate prior and posterior hazard ratio (HR) samples for PFS and OS.

    Returns:
        dict: A nested dictionary containing arrays of HR samples.
    """
    from numpy.random import default_rng

    # RNG seeded for reproducibility of the "prior" samples
    rng = default_rng(seed_for_prior)
    # Dictionary to hold final results
    res = {}

    for outcome in ["pfs", "os"]:
        params = hr_log_transform(
            hr=hr_params[outcome]["hr"],
            hr_lower=hr_params[outcome]["low"],
            hr_upper=hr_params[outcome]["high"]
        )
        mu = params["mu_log_hr"]
        sigma = params["sigma_log_hr"]
        log_samples = rng.normal(loc=mu, scale=sigma, size=n_psa)
        hr_samples = np.exp(log_samples)
        res[outcome] = hr_samples



    return res

hr_vecs = get_hr_vec(hr_params=hr_params, n_psa=n_psa, seed_for_prior=seed)

In [ ]:
def simulate_survival_vectorized(cycle_range, intercept, log_scale, hr=None):
    """
    Computes survival probabilities for each cycle.

    If hr is None, computes the control survival curve (1D array over cycles).
    If hr is provided and is a 1D array of hazard ratios (shape = (n_draws,)),
    returns an array of survival curves with shape (n_draws, len(cycle_range)).

    Parameters
    ----------
    cycle_range : array-like
        Sequence of cycle indices.
    intercept : float
        Intercept parameter for the underlying survival model.
    log_scale : float
        Log-scale parameter of the survival model.
    hr : array-like or None
        Hazard ratio(s) for proportional hazards. If array, must be 1D of length n_draws.

    Returns
    -------
    np.ndarray
        If hr is None, shape (n_cycles,).
        If hr is array, shape (n_draws, n_cycles).
    """
    adjusted_cycle = cycle_range * (3/4)  # shape (n_cycles,)
    # Control survival curve
    surv_ctrl = 1 - norm.cdf((np.log(adjusted_cycle + 1e-10) - intercept) / np.exp(log_scale))
    surv_ctrl[-1] = 0  # enforce last cycle = 0

    if hr is None:
        return surv_ctrl
    else:
        # hr should be hazard ratios; shape (n_draws,)
        surv = surv_ctrl[None, :] ** hr[:, None]  # broadcasting
        surv[:, -1] = 0  # enforce boundary condition
        return surv

In [ ]:
def get_nmb_vectorized(
    hr_vecs: dict = hr_vecs,
    wtp: float = wtp,
    case_params: dict = case_params,
    utility: dict = utility,
    cost: dict = cost,
    price_drug: float = point_price,
    cycle_range: np.ndarray = cycle_range,
    discount_factor: np.ndarray = discount_factor
) -> dict:
    """
    Vectorized net monetary benefit (NMB) for chemo vs combo arms.

    Parameters
    ----------
    hr_pfs_vec : array-like
        1-D array of PFS hazard ratio draws (combo vs chemo).
    hr_os_vec : array-like
        1-D array of OS hazard ratio draws (combo vs chemo).
    case_params : dict
        {"pfs_chemo": {"intercept", "log_scale"},
         "os_chemo":  {"intercept", "log_scale"}}
    price_drug : float
        Unit price of the combo drug.
    wtp : float
        Willingness-to-pay per QALY.
    utility : dict
        {"stable": float, "prog": float}
    cost : dict
        Cost parameters including
        "monitoring_stable", "monitoring_prog",
        "admin_first", "admin_sub", "terminal",
        "intro_cost", "maintain_cost_chemo",
        "dose_drug"
    cycle_range : array-like
        Sequence of cycle indices.
    discount_factor : array-like
        Per-cycle discounting multipliers.

    Returns
    -------
    dict
        {
          "chemo": (qalys_chemo, costs_chemo, NB_chemo),
          "combo": (qalys_combo, costs_combo, NB_combo),
          "delta": (delta_qaly, delta_cost, delta_NB)
        }
    """

    # 1. Generate PFS and OS HR sample vectors
    hr_pfs_vec = hr_vecs["pfs"]
    hr_os_vec  = hr_vecs["os"]

    n_cycles = len(cycle_range)
    cycle_factor = 0.75 / 12.0

    # --- Chemo arm (scalar) ---
    pfs_c = simulate_survival_vectorized(cycle_range,
                                         intercept=case_params["pfs_chemo"]["intercept"],
                                         log_scale=case_params["pfs_chemo"]["log_scale"],
                                         hr=None)
    os_c  = simulate_survival_vectorized(cycle_range,
                                         intercept=case_params["os_chemo"]["intercept"],
                                         log_scale=case_params["os_chemo"]["log_scale"],
                                         hr=None)
    prog_c   = np.maximum(os_c - pfs_c, 0.0)
    stable_c = pfs_c.copy()
    dead_c   = 1.0 - os_c
    stable_c[-1] = 0.0
    prog_c[-1]   = 0.0
    dead_c[-1]   = 1.0

    qalys_chemo = np.sum(
        (stable_c * utility["stable"] + prog_c * utility["prog"]) * cycle_factor * discount_factor
    )
    monitoring_c = cost["monitoring_stable"] * stable_c + cost["monitoring_prog"] * prog_c
    admin_c      = np.zeros(n_cycles)
    admin_c[:37] = stable_c[:37] * (cost["admin_first"] + cost["admin_sub"])
    final_c      = np.concatenate(([dead_c[0]], np.diff(dead_c))) * cost["terminal"]
    treat_c      = np.zeros(n_cycles)
    treat_c[:4]  = cost["intro_cost"] * stable_c[:4]
    treat_c[4:36]= cost["maintain_cost_chemo"] * stable_c[4:36]
    costs_chemo  = np.sum((monitoring_c + admin_c + final_c + treat_c) * discount_factor)
    NB_chemo     = wtp * qalys_chemo - costs_chemo

    # --- Combo arm (vectorized) ---
    pfs_x = simulate_survival_vectorized(cycle_range,
                                         intercept=case_params["pfs_chemo"]["intercept"],
                                         log_scale=case_params["pfs_chemo"]["log_scale"],
                                         hr=hr_pfs_vec)
    os_x  = simulate_survival_vectorized(cycle_range,
                                         intercept=case_params["os_chemo"]["intercept"],
                                         log_scale=case_params["os_chemo"]["log_scale"],
                                         hr=hr_os_vec)
    pfs_x[:, -1] = 0.0
    os_x[:, -1]  = 0.0
    prog_x   = np.maximum(os_x - pfs_x, 0.0)
    stable_x = pfs_x
    dead_x   = 1.0 - os_x
    dead_x[:, -1] = 1.0

    qalys_combo = np.sum(
        (stable_x * utility["stable"] + prog_x * utility["prog"]) * cycle_factor * discount_factor,
        axis=1
    )
    monitoring_x = cost["monitoring_stable"] * stable_x + cost["monitoring_prog"] * prog_x
    admin_x      = np.zeros_like(stable_x)
    admin_x[:, :37] = stable_x[:, :37] * (cost["admin_first"] + cost["admin_sub"])
    final_x      = np.hstack([dead_x[:, :1], np.diff(dead_x, axis=1)]) * cost["terminal"]
    drug_intro   = cost["dose_drug"] * price_drug + cost["intro_cost"]
    drug_maint   = cost["dose_drug"] * price_drug + cost["maintain_cost_chemo"]
    treat_x      = np.zeros_like(stable_x)
    treat_x[:, :4]  = stable_x[:, :4] * drug_intro
    treat_x[:, 4:36]= stable_x[:, 4:36] * drug_maint
    costs_combo = np.sum((monitoring_x + admin_x + final_x + treat_x) * discount_factor, axis=1)
    NB_combo    = wtp * qalys_combo - costs_combo

    # --- Differences ---
    delta_qaly = qalys_combo - qalys_chemo
    delta_cost = costs_combo - costs_chemo
    delta_NB   = NB_combo   - NB_chemo

    res = {
        "qalys": {
            "chemo": np.mean(qalys_chemo),
            "combo": np.mean(qalys_combo)
        },
        "costs": {
            "chemo": np.mean(costs_chemo),
            "combo": np.mean(costs_combo)
        },
        "NB": {
            "chemo": np.mean(NB_chemo),
            "combo": np.mean(NB_combo)
        },
        "delta": {
            "qaly": np.mean(delta_qaly),
            "cost": np.mean(delta_cost),
            "NB_mean": np.mean(delta_NB),
            "NB_se": np.std(delta_NB)
        }
    }

    return res

In [ ]:
# get_nmb_vectorized()

In [ ]:
def get_trial_update_nmb(
    a_value: float = a_value,
    hr_vecs: dict = hr_vecs,
    sample_size_now: int = sample_size_now,
    sample_size_new: int = sample_size_new,
    n_psa: int = n_psa,
    seed_for_prior: int = seed,
    price_drug: float = point_price,
    wtp: float = wtp,
    case_params: dict = case_params,
    utility: dict = utility,
    cost: dict = cost,
    cycle_range: np.ndarray = cycle_range,
    discount_factor: np.ndarray = discount_factor
) -> dict:
    """
    Computes prior and posterior net monetary benefit (NMB) updates given new trial sample sizes.

    Returns
    -------
    dict
        {
          "prior": np.ndarray of shape (n_psa,),
          "posterior": np.ndarray of shape (n_psa, n_psa)
        }
    """

    from numpy.random import default_rng

    # 1. Obtain mean and SE of delta NMB from prior PSA
    nmb_out = get_nmb_vectorized(
        hr_vecs=hr_vecs,
        case_params=case_params,
        price_drug=price_drug,
        wtp=wtp,
        utility=utility,
        cost=cost,
        cycle_range=cycle_range,
        discount_factor=discount_factor
    )

    nb_mean = nmb_out["delta"]["NB_mean"]
    nb_se   = nmb_out["delta"]["NB_se"]
    # nb_se   = np.sqrt(nmb_out["delta"]["NB_se"]**2 / a_value)

    # 2. Draw PRIOR samples of delta-NMB
    rng = default_rng(seed_for_prior)
    nb_prior = rng.normal(loc=nb_mean, scale=np.sqrt(nb_se**2 / a_value), size=n_psa)

    # 3. Precompute variances and precisions for Bayesian update
    prior_prec = 1.0 / (nb_se**2 / a_value)
    pop_var     = ((nb_se**2) * sample_size_now) / 2
    sample_var  = 2 * pop_var / sample_size_new
    sample_prec = 1.0 / sample_var
    post_prec   = prior_prec + sample_prec
    post_sd     = np.sqrt(1.0 / post_prec)

    # 4. POSTERIOR: update each prior draw with hypothetical new data
    posterior = np.zeros((n_psa, n_psa))
    for i, mu_prior in enumerate(nb_prior):
        new_data = rng.normal(loc=mu_prior, scale=np.sqrt(sample_var), size=sample_size_new)
        Xbar     = new_data.mean()
        post_mean = (prior_prec * nb_mean + sample_prec * Xbar) / post_prec
        posterior[i, :] = rng.normal(loc=post_mean, scale=post_sd, size=n_psa)

    return {"prior": nb_prior, "posterior": posterior}

In [ ]:
# get_trial_update_nmb(n_psa=n_psa)

In [ ]:
def get_ev(
    a_value: float = a_value,
    sample_size_now: int = sample_size_now,
    sample_size_new: int = sample_size_new,
    n_psa: int = n_psa,
    seed_for_prior: int = seed,
    price_drug: float = point_price,
    wtp: float = wtp,
    case_params: dict = case_params,
    hr_vecs: dict = hr_vecs,
    utility: dict = utility,
    cost: dict = cost,
    cycle_range: np.ndarray = cycle_range,
    discount_factor: np.ndarray = discount_factor
) -> dict:

    # 1. Draw PFS/OS hazard‐ratio vectors
    hr_pfs_vec = hr_vecs["pfs"]
    hr_os_vec  = hr_vecs["os"]

    # 2. Base‐case NMB for chemo vs combo
    nmb_out   = get_nmb_vectorized(
        hr_vecs=hr_vecs,
        case_params=case_params,
        price_drug=price_drug,
        wtp=wtp,
        utility=utility,
        cost=cost,
        cycle_range=cycle_range,
        discount_factor=discount_factor
    )
    # unpack NB from the returned tuples
    NB_chemo  = nmb_out["NB"]["chemo"]
    # NB_combo  = nmb_out["NB"]["combo"]

    EV_ct_pp  = NB_chemo
    # EV_nt_pp  = NB_combo

    # 3. Perform trial‐update to get prior and posterior NMB draws
    trial     = get_trial_update_nmb(
        a_value=a_value,
        hr_vecs=hr_vecs,
        sample_size_now=sample_size_now,
        sample_size_new=sample_size_new,
        n_psa=n_psa,
        seed_for_prior=seed_for_prior,
        price_drug=price_drug,
        wtp=wtp,
        case_params=case_params,
        utility=utility,
        cost=cost,
        cycle_range=cycle_range,
        discount_factor=discount_factor
    )

    prior_arr     = trial["prior"]       # shape (n_psa,)
    posterior_mat = trial["posterior"]   # shape (n_psa, n_psa)

    # 4. Compute expected posterior by draw and EVSI
    post_exp      = posterior_mat.mean(axis=1)               # E[NB|data] per prior draw
    EV_update_pp  = np.mean(np.maximum(0, post_exp)) # EVSI

    EV_update_pp = EV_delta_nb_pp + EV_ct_pp

    EV_nt_pp  = np.mean(prior_arr) + EV_ct_pp

    return {
        "EV_ct_pp": EV_ct_pp,
        "EV_nt_pp": EV_nt_pp,
        "EV_update_pp": EV_update_pp
    }

In [ ]:
# a_value from 0.1 to 1.0 in steps of 0.1
a_value_arr = np.arange(0.1, 1.0 + 0.1, 0.1)

# price_drug from 70% to 130% of the base price, in 10%-point increments
price_drug_arr = value_based_price * np.arange(1 - 0.3, 1 + 0.3, 0.05)

In [ ]:
!pip install tqdm joblib tqdm_joblib
import multiprocessing
import itertools
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib

# Number of parallel jobs: using one less than the number of CPUs
n_jobs = 2 # multiprocessing.cpu_count() # 8 in total

# The run_combination function remains the same:
def run_combination(a_val, drug_price):
    """Runs a single combination and returns the computed EV values."""
    out = get_ev(
        a_value=a_val,
        price_drug=drug_price
    )
    return {
        "a_value": a_val,
        "price_drug": drug_price,
        "EV_ct_pp": out['EV_ct_pp'],
        "EV_nt_pp": out['EV_nt_pp'],
        "EV_update_pp": out['EV_update_pp']
    }

/usr/local/lib/python3.11/dist-packages/tqdm_joblib/__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [ ]:
# Create all combinations of a_value and drug_price
combinations = list(itertools.product(a_value_arr, price_drug_arr))

# Use tqdm_joblib for progress bar with parallel processing
with tqdm_joblib(tqdm(desc="Running combinations", total=len(combinations))):
    results = Parallel(n_jobs=n_jobs, pre_dispatch="2*n_jobs")(
        delayed(run_combination)(a_val, drug_price)
        for a_val, drug_price in combinations
    )

# Create a DataFrame from the results
df_a_price = pd.DataFrame(results)

Running combinations:   0%|          | 0/130 [00:00<?, ?it/s]

  0%|          | 0/130 [00:00<?, ?it/s]

In [ ]:
enbs_paths = "/content/drive/MyDrive/Colab Notebooks/01_Research/01_Local_Confirmatory_RCT/03_output/03_enbs"

df_a_price.to_csv(f"{enbs_paths}/01_combination_a_unit_price.csv", index=False)